# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [2]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [3]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('KOLOSAL_API_KEY')

if api_key and api_key.startswith('kol_') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'Llama 4 Maverick'
openai = OpenAI(
    api_key= os.getenv('KOLOSAL_API_KEY'),
    base_url= 'https://api.kolosal.ai/v1'
)

API key looks good so far


In [4]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [5]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [6]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [7]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/
https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/
https://edwarddonner.com/2025/11/11/

In [8]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [9]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'curriculum page', 'url': 'https://edwarddonner.com/curriculum/'}]}

In [10]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [11]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling Llama 4 Maverick
Found 2 relevant links


{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'curriculum/courses page',
   'url': 'https://edwarddonner.com/curriculum/'}]}

In [12]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling Llama 4 Maverick
Found 6 relevant links


{'links': [{'type': 'about page', 'url': 'https://huggingface.co/allenai'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'company page', 'url': 'https://huggingface.co/huggingface'},
  {'type': 'brand page', 'url': 'https://huggingface.co/brand'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [13]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [14]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling Llama 4 Maverick
Found 4 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Community
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
Lightricks/LTX-2
Updated
about 23 hours ago
•
188k
•
625
tencent/HY-MT1.5-1.8B
Updated
8 days ago
•
7.29k
•
681
Qwen/Qwen-Image-2512
Updated
9 days ago
•
18.4k
•
535
IQuestLab/IQuest-Coder-V1-40B-Loop-Instruct
Updated
2 days ago
•
9.65k
•
293
MiniMaxAI/MiniMax-M2.1
Updated
13 days ago
•
200k
•
965
Browse 2M+ models
Spaces
Running
Featured
4k
Wan2.2 Animate
👁
4k
Wan2.2 Animate
Running
on
Zero
1.18k
Z Image Turbo
🖼
1.18k
Generate stunning images from text descriptions in seconds
Running
on
Zero
MCP
Featured
268
Qwen-Image-Edit-251

In [15]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [16]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [17]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling Llama 4 Maverick
Found 5 relevant links


"\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nCommunity\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nLightricks/LTX-2\nUpdated\nabout 23 hours ago\n•\n188k\n•\n625\ntencent/HY-MT1.5-1.8B\nUpdated\n8 days ago\n•\n7.29k\n•\n681\nQwen/Qwen-Image-2512\nUpdated\n9 days ago\n•\n18.4k\n•\n535\nIQuestLab/IQuest-Coder-V1-40B-Loop-Instruct\nUpdated\n2 days ago\n•\n9.65k\n•\n293\nMiniMaxAI/MiniMax-M2.1\nUpdated\n13 days ago\n•\n200k\n•\n965\nBrowse 2M+ models\nSpaces\nRunning\nFeatured\n4k\nWan2.2 An

In [18]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="Llama 4 Maverick",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [19]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling Llama 4 Maverick
Found 4 relevant links


# Hugging Face: The AI Community Building the Future
Hugging Face is a collaboration platform for the machine learning community to build, share, and deploy AI models, datasets, and applications. The company provides a range of tools and services for individuals, teams, and enterprises to accelerate their AI development.

## Company Culture
Hugging Face is driven by a community-first approach, fostering collaboration and innovation among its users. The platform is open to everyone, allowing users to create, discover, and share machine learning models, datasets, and applications.

## Products and Services
* **Models**: A vast repository of over 2 million machine learning models, allowing users to explore, download, and deploy models for various applications.
* **Datasets**: A collection of over 500,000 datasets, providing users with a vast resource for training and fine-tuning their models.
* **Spaces**: A platform for hosting and collaborating on machine learning applications, with features like version control and deployment options.
* **Enterprise**: A range of enterprise-grade solutions, including dedicated support, access controls, and Single Sign-On (SSO) integration, designed to help organizations scale their AI development.

## Careers
Hugging Face is committed to building a talented team of professionals passionate about AI and machine learning. Current job openings are available on the company's careers page.

## Customers
Hugging Face's platform is used by a wide range of customers, from individual developers to large enterprises. Notable users include Facebook AI (Meta), who utilize the platform for their research and development.

## Benefits
* **Accelerate AI Development**: Hugging Face's platform and tools help users build, deploy, and iterate on AI models and applications faster.
* **Collaborate with the Community**: Join a vibrant community of machine learning professionals, researchers, and developers to share knowledge, models, and datasets.
* **Enterprise-Grade Solutions**: For organizations, Hugging Face offers a range of enterprise-grade features, including security, access controls, and dedicated support.

## Pricing
Hugging Face offers a range of pricing plans, including a free tier, Team plans starting at $20/user/month, and custom Enterprise plans. Contact the sales team for more information on Enterprise solutions.

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [20]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="MiniMax M2",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [21]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling Llama 4 Maverick
Found 6 relevant links


# Hugging Face – The AI Community Building the Future  

**Vision:** To make cutting‑edge AI accessible to everyone by fostering a collaborative, open‑source ecosystem where developers, researchers, and enterprises can create, share, and deploy models together.  

---  

## Community‑First Platform  

- **2 M+ Models** – From tiny language models to massive multimodal systems, the Hub hosts a rapidly growing library of pre‑trained and fine‑tunable models.  
- **500 k+ Datasets** – Curated and community‑contributed data collections covering every domain, from scientific research to industry benchmarks.  
- **1 M+ Spaces (AI Apps)** – Ready‑to‑run, shareable applications that let users try out models instantly in the browser.  

These figures are constantly refreshed; recent highlights include **Lightricks/LTX‑2** (≈188 k downloads) and **MiniMaxAI/MiniMax‑M2.1** (≈200 k downloads), underscoring the platform’s high‑velocity adoption.  

---  

## What We Offer  

| Offering | Description | Who Benefits |
|----------|-------------|--------------|
| **Open‑Source Stack** | Core libraries (Transformers, Diffusers, Datasets, Accelerate) – all fully open‑source. | Developers & researchers building custom pipelines. |
| **Public Hub** | Free hosting for unlimited public models, datasets, and Spaces. | Community contributors, students, and hobbyists. |
| **Compute & Enterprise** | Managed GPU clusters, scalable training & inference, enterprise‑grade security & support. | Startups, enterprises, and research labs needing reliable, high‑performance AI infrastructure. |
| **Team & Collaboration Tools** | Role‑based access, versioned releases, workflow integrations (Git, CI/CD). | Multi‑disciplinary teams and organizations. |
| **Pricing & Plans** | Flexible tiers from individual developers to large enterprises (details on pricing page). | All budgets and scales. |

---  

## Who Uses Hugging Face?  

- **Researchers & Academics** – Publish models and datasets, replicate experiments, and accelerate scientific discovery.  
- **Startups & Scale‑Ups** – Prototype, iterate, and deploy AI products without heavy infrastructure investment.  
- **Enterprise Teams** – Build production‑grade solutions, enforce governance, and manage cost‑effective compute.  
- **Educators & Students** – Learn, teach, and showcase AI work in a community‑driven environment.  

Our **76 k+** AI & ML‑interested followers represent a vibrant, global community of creators and adopters.  

---  

## Company Culture  

- **Open‑Source First:** Every core component is released under permissive licenses, inviting the community to inspect, improve, and extend the technology.  
- **Collaboration Over Competition:** Shared workspaces, versioned releases, and transparent activity feeds encourage joint development rather than siloed work.  
- **Impact‑Driven:** Our mission is to democratize AI—making powerful tools available to anyone, anywhere.  
- **Remote‑Friendly & Global:** Team members work from diverse locations, reflecting the truly worldwide nature of our user base.  

---  

## Careers at Hugging Face  

We’re expanding fast and looking for talented people across engineering, research, product, community, and operations.  

- **Open Positions:** Check our **Current Openings** page for roles in machine‑learning engineering, platform development, DevOps, product management, and community outreach.  
- **What We Look For:** Passion for open‑source AI, ability to collaborate in a distributed environment, and a drive to shape the future of machine learning.  
- **Why Join:** Work on cutting‑edge AI infrastructure used by millions, enjoy flexible remote work, and have a direct impact on the community’s success.  

---  

## Why Choose Hugging Face?  

1. **Scale & Depth** – Access millions of ready‑to‑use resources, all catalogued and searchable.  
2. **Speed to Insight** – Spin up a model or a demo in minutes; no heavy setup required.  
3. **Enterprise‑Grade Reliability** – Managed compute, security, and support built for production workloads.  
4. **Community Power** – Benefit from an active network of 76 k+ AI enthusiasts who constantly share updates, tutorials, and best practices.  

---  

**Get Started Today**  

- Explore the **2 M+** models, **500 k+** datasets, and **1 M+** Spaces.  
- Sign up for a free account or contact **Enterprise Sales** for custom solutions.  

**Hugging Face – Where the AI community co‑creates the future.**

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>